# 01. Scaled Dot-Product Attention 기초

## 학습 목표

- Q, K, V와 score·weight·output의 shape를 추적한다.
- 안정적인 softmax와 `sqrt(d_k)` scaling을 구현한다.
- padding mask가 softmax 전에 적용되어야 하는 이유를 확인한다.

실행 위치: 이 notebook이 있는 `attention-is-all-you-need/` 폴더. 외부 dependency는 NumPy 하나다.

## 핵심 문법

| 문법 | 의미 |
|---|---|
| `array @ array.T` | 마지막 두 축에 matrix multiplication을 적용한다. |
| `axis=-1` | 마지막 축, 즉 key 위치 방향으로 연산한다. |
| `keepdims=True` | broadcasting이 가능하도록 축 크기 1을 유지한다. |
| `np.where` | mask 조건에 따라 score 또는 큰 음수를 선택한다. |
| `assert` | shape와 확률 합 같은 불변식이 깨지면 즉시 중단한다. |

In [ ]:
# numpy는 vector와 matrix 연산을 제공한다.
import numpy as np

# 출력 자릿수를 줄여 attention matrix를 읽기 쉽게 만든다.
np.set_printoptions(precision=4, suppress=True)

# 같은 입력에서 항상 같은 결과를 얻도록 random generator seed를 고정한다.
rng = np.random.default_rng(seed=170603762)

In [ ]:
# 마지막 축에 수치적으로 안정적인 softmax를 적용하는 함수다.
def stable_softmax(scores: np.ndarray) -> np.ndarray:
    # 각 행의 최댓값을 빼 exp overflow와 saturation 위험을 줄인다.
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    # 이동한 score를 양수 지수값으로 변환한다.
    exponentials = np.exp(shifted)
    # key 방향 합으로 나눠 각 query 행의 확률 합을 1로 만든다.
    return exponentials / np.sum(exponentials, axis=-1, keepdims=True)

# 논문 식 (1)을 mask까지 포함해 구현한다.
def scaled_dot_product_attention(
    queries: np.ndarray,
    keys: np.ndarray,
    values: np.ndarray,
    allowed: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    # Q와 K의 feature 차원 d_k는 dot product를 위해 같아야 한다.
    assert queries.shape[-1] == keys.shape[-1]
    # K의 sequence 길이와 V의 sequence 길이는 같은 위치를 나타내야 한다.
    assert keys.shape[-2] == values.shape[-2]
    # d_k를 float로 읽어 sqrt scaling denominator를 만든다.
    d_k = float(keys.shape[-1])
    # K의 마지막 두 축을 바꿔 모든 query-key dot product를 한 번에 계산한다.
    scores = queries @ np.swapaxes(keys, -2, -1)
    # score 분산이 d_k에 따라 커지는 효과를 완화한다.
    scores = scores / np.sqrt(d_k)
    # allowed가 주어졌을 때만 padding 또는 causal mask를 적용한다.
    if allowed is not None:
        # 허용되지 않은 연결은 softmax 뒤 확률이 0에 가까워지도록 큰 음수로 바꾼다.
        scores = np.where(allowed, scores, -1.0e9)
    # key 위치 방향으로 합이 1인 attention weight를 만든다.
    weights = stable_softmax(scores)
    # 각 query가 value vector의 가중합을 모은 context를 계산한다.
    output = weights @ values
    # 계산 결과와 관찰용 weight를 함께 반환한다.
    return output, weights

In [ ]:
# query 세 개를 만들며 각 query의 feature 차원 d_k는 4다.
q = rng.normal(size=(3, 4))
# key도 세 위치와 같은 d_k=4를 사용한다.
k = rng.normal(size=(3, 4))
# value는 key마다 전달할 정보이며 여기서는 d_v=2다.
v = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])

# mask 없는 scaled dot-product attention을 실행한다.
context, weights = scaled_dot_product_attention(q, k, v)
# 중간 tensor의 shape를 출력해 수식과 code를 연결한다.
print('Q, K, V shapes:', q.shape, k.shape, v.shape)
# 각 query가 key 세 개에 부여한 확률을 출력한다.
print('attention weights:\n', weights)
# value의 가중합 결과를 출력한다.
print('context:\n', context)

# score matrix는 query 수 × key 수여야 한다.
assert weights.shape == (3, 3)
# output은 query 수 × value 차원이어야 한다.
assert context.shape == (3, 2)
# 각 query 행의 확률 합은 부동소수점 오차 안에서 1이어야 한다.
assert np.allclose(weights.sum(axis=-1), 1.0)

## Scaling 효과

차원이 커질 때 scaling이 없으면 score 절댓값이 커져 softmax가 한 위치에 과도하게 집중할 수 있다. entropy가 작을수록 분포가 더 뾰족하다.

In [ ]:
# 확률 분포의 평균 entropy를 계산한다.
def mean_entropy(probabilities: np.ndarray) -> float:
    # log(0)을 피하도록 machine-safe한 작은 하한을 적용한다.
    safe = np.clip(probabilities, 1.0e-12, 1.0)
    # 각 행 entropy를 구한 뒤 모든 query의 평균을 Python float로 반환한다.
    return float(np.mean(-np.sum(safe * np.log(safe), axis=-1)))

# d_k=256인 query와 key를 128개씩 만들어 차원이 큰 조건을 만든다.
large_q = rng.normal(size=(128, 256))
# 같은 분포의 key를 별도로 생성한다.
large_k = rng.normal(size=(128, 256))
# scaling 전 raw dot-product score다.
raw_scores = large_q @ large_k.T
# 논문 방식대로 sqrt(d_k)로 나눈 score다.
scaled_scores = raw_scores / np.sqrt(large_q.shape[-1])
# 두 score를 각각 softmax probability로 바꾼다.
raw_probabilities = stable_softmax(raw_scores)
# scaling된 probability도 같은 방식으로 계산한다.
scaled_probabilities = stable_softmax(scaled_scores)
# scaling 전후 entropy를 비교한다.
print('without scaling entropy:', mean_entropy(raw_probabilities))
# scaling 뒤 분포가 덜 포화되는지 확인한다.
print('with scaling entropy   :', mean_entropy(scaled_probabilities))
# 이 seed와 조건에서는 scaling 뒤 평균 entropy가 더 커야 한다.
assert mean_entropy(scaled_probabilities) > mean_entropy(raw_probabilities)

## Padding mask

세 번째 key가 padding이라고 가정한다. mask는 softmax 뒤 weight를 0으로 덮는 방식보다 softmax 전에 적용하는 편이 안전하다. 뒤에서 0만 곱하면 남은 확률의 합을 다시 정규화해야 하기 때문이다.

In [ ]:
# True는 연결 허용, False는 padding key 차단을 뜻한다.
padding_allowed = np.array([[True, True, False]])
# broadcasting으로 같은 key mask를 query 세 개 모두에 적용한다.
masked_context, masked_weights = scaled_dot_product_attention(
    q,
    k,
    v,
    allowed=padding_allowed,
)
# 세 번째 key의 weight가 모든 query에서 0인지 출력한다.
print('masked attention weights:\n', masked_weights)
# 큰 음수 mask 뒤 softmax 결과는 수치 오차 안에서 0이어야 한다.
assert np.allclose(masked_weights[:, 2], 0.0)
# 허용된 두 key의 확률 합은 여전히 1이어야 한다.
assert np.allclose(masked_weights[:, :2].sum(axis=-1), 1.0)
# masked output도 query 수 × d_v shape를 유지한다.
assert masked_context.shape == (3, 2)

## 통과 기준과 다음 실습

- 모든 assertion이 통과한다.
- `weights.shape == (query_length, key_length)`를 설명할 수 있다.
- scaling이 score 평균을 0으로 만드는 것이 아니라 분산 크기를 조절한다는 점을 설명할 수 있다.
- 다음은 [02_practice.ipynb](02_practice.ipynb)에서 causal mask와 multi-head reshape를 구현한다.